# EDA - Dataset de Churn (Base de Treino)

Analise exploratoria do dataset agregado por VIN (`dataset_churn_pos_venda.csv`), usado para o treinamento dos modelos de Churn e Perfil.

In [ ]:
import matplotlib
matplotlib.use('Agg')

import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Fix CWD
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    os.chdir(notebook_dir.parent)
print(f"CWD: {os.getcwd()}")

os.makedirs('reports', exist_ok=True)
sns.set_theme(style='whitegrid')

## Carregamento e visao geral

In [ ]:
base_path = 'data/processed/dataset_churn_pos_venda.csv'
if not os.path.exists(base_path):
    print(f"AVISO: {base_path} nao encontrado. Rode o pipeline primeiro.")
else:
    df = pd.read_csv(base_path)
    print(f'Linhas: {df.shape[0]:,}')
    print(f'Colunas: {df.shape[1]:,}')
    display(df.head())

## Distribuicao de Churn (Real)

In [ ]:
from src.pipeline.config import TARGET_CHURN

if TARGET_CHURN in df.columns:
    plt.figure(figsize=(7, 5))
    churn_counts = df[TARGET_CHURN].value_counts(normalize=True).sort_index().mul(100)
    sns.barplot(x=churn_counts.index, y=churn_counts.values, hue=churn_counts.index, palette='RdYlGn_r', legend=False)
    plt.title('Distribuicao de Churn Futuro 18m')
    plt.ylabel('Percentual (%)')
    plt.xlabel('Classe')
    plt.xticks([0, 1], ['No Churn', 'Churn'])
    for i, v in enumerate(churn_counts.values):
        plt.text(i, v + 0.5, f'{v:.1f}%', ha='center')
    plt.savefig('reports/base2_distribuicao_churn.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Coluna target ausente: {TARGET_CHURN}')


## Correlacao de Features Comportamentais com o Target

In [ ]:
from src.pipeline.config import SNAPSHOT_FEATURES_NUMERIC, TARGET_CHURN

behav_cols = [c for c in SNAPSHOT_FEATURES_NUMERIC if c in df.columns]
corr_cols = behav_cols + [TARGET_CHURN]
corr = df[corr_cols].corr(numeric_only=True)[TARGET_CHURN].drop(TARGET_CHURN).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=corr.values, y=corr.index, hue=corr.index, palette='coolwarm', legend=False)
plt.title('Correlacao de Features Comportamentais com Churn Futuro 18m')
plt.xlabel('Coeficiente de Correlacao')
plt.savefig('reports/base2_correlacao_churn.png', dpi=150, bbox_inches='tight')
plt.show()
